# Session Control — Claude Managed Agents

Managed Agents sessions are built for long work: minutes or hours of tool calls, with your client coming and going. This module covers the controls you need around such a session:

| Control | How |
|---|---|
| **Cap the spend** | Create the session with a `budget`; watch `session.usage` on the stream |
| **Add context mid-session** | Send a `system.message` |
| **Change direction** | Send `user.interrupt`, then a new `user.message` |
| **Survive a dropped connection** | Reopen the stream and catch up from the event history |
| **Audit what happened** | List past events, filtered by type |
| **Pause at the cap, then resume** | `stop_reason: budget_reached`; raise the budget with `sessions.update()` |
| **Inspect and close** | `sessions.retrieve()`, `list()`, `archive()` |

All of this matters even more in modules 06–07, where one session runs many agents at once.

## 1. Setup

In [1]:
%pip install -q -U "anthropic>=1.13" python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import time

import anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

client = anthropic.Anthropic()

environment = client.beta.environments.create(
    name="session-control-env",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
)
agent = client.beta.agents.create(
    name="Session Control Agent",
    model="claude-opus-5-5",
    system="You are a helpful engineer working in a Linux sandbox. Keep replies short.",
    tools=[{"type": "agent_toolset_20260401"}],
)
print(f"Environment : {environment.id}")
print(f"Agent       : {agent.id}  (v{agent.version})")

Environment : env_01Bbx5DjUfem5gyAUbMUyZMV
Agent       : agent_01Hpiobf4sXNVFcDRUTmP7xV  (v1)


## 2. Create a Session with a Budget

A **session budget** is a hard spending cap you can only attach **when the session is created**. The platform prices everything the session consumes at public list rates — its **list cost**:

- model tokens, at each model's list price
- web searches, at $10 per 1,000
- session running time, at $0.08 per hour

When the list cost reaches the cap, the session **pauses** (it doesn't terminate). You can raise, lower, or remove the cap later.

`amount` is a whole number of **US cents written as a string**: `"200"` is $2.00. Decimals like `"2.00"` are rejected.

In [3]:
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    title="Session control",
    budget={"type": "limit", "max_list_cost": {"amount": "200", "currency": "USD"}},
)

print(f"Session ID : {session.id}")
print(f"Status     : {session.status}")
print(f"Budget     : {session.budget.max_list_cost.amount} cents")

Session ID : sesn_01Kk66ZmWsXn8GFjCzBLxQff
Status     : idle
Budget     : 200 cents


## 3. A `follow()` Helper

The turn loop from module 03, reshaped as a reusable function. It takes an **already-open** stream — so you can send events however you like, stream-first — and returns the final `stop_reason`. It also prints a few events we haven't looked at yet:

| Event | Meaning |
|---|---|
| `user.message` | **Echoed when the agent picks it up.** A message you send while the agent is busy is queued, and its `processed_at` stays null until then. |
| `user.interrupt` | Your interrupt was applied |
| `session.usage` | Cumulative usage and list cost, emitted right before every idle. It also echoes the budget. |
| `session.error` | Something went wrong; it carries a typed `error` |

`after_text` handles the one subtle case, interrupts, explained in section 5.

In [4]:
def text_of(event) -> str:
    return "".join(block.text for block in event.content if block.type == "text")


def cents(amount) -> str:
    return f"${int(amount.amount) / 100:.2f}"


def follow(stream, *, after_text=None, on_event=None) -> str:
    """Print events from an open stream until the turn ends; return the stop_reason type.

    after_text: an end_turn idle only counts once a user.message with this text has
                been processed (used after an interrupt).
    on_event:   optional callback run on every event (used to trigger an interrupt).
    """
    waiting_for = after_text
    for event in stream:
        if on_event:
            on_event(event)
        match event.type:
            case "user.message":
                print(f"[user] {text_of(event)[:80]}")
                if text_of(event) == waiting_for:
                    waiting_for = None
            case "user.interrupt":
                print("[interrupt applied]")
            case "agent.message":
                print(f"Agent: {text_of(event)}")
            case "agent.tool_use":
                print(f"[{event.name}] {str(event.input.get('command') or event.input.get('file_path') or '')[:90]}")
            case "session.usage":
                budget = f" of {cents(event.budget.max_list_cost)}" if event.budget else ""
                print(f"[usage] {cents(event.usage.list_cost)}{budget}, {event.usage.active_seconds:.0f}s active")
            case "session.error":
                print(f"[error] {event.error.message if event.error else event}")
            case "session.status_idle":
                reason = event.stop_reason.type
                if reason == "requires_action":
                    continue
                if reason == "end_turn" and waiting_for:
                    print("[idle: end_turn — the interrupted turn; waiting for the redirect]")
                    continue
                print(f"[idle: {reason}]")
                return reason
            case "session.status_terminated":
                print("[terminated]")
                return "terminated"


def message(text: str) -> dict:
    return {"type": "user.message", "content": [{"type": "text", "text": text}]}

## 4. Add Context Mid-Session with `system.message`

A **`system.message`** event appends privileged, system-level context to the session — for this turn and every later one. It doesn't replace the agent's `system` prompt; it adds to it. Use it for context you learn at runtime: the user's timezone, a changed constraint, a fetched policy.

Send it **in the same request** as the `user.message` it should inform, and put it **last**: a `system.message` that isn't the final event in the request is rejected with a 400 error. (Supported on recent models, including `claude-opus-5-5`.)

In [5]:
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[
            message("What's my local time right now? Check the sandbox clock with bash."),
            # system.message must be the last event in the request
            {"type": "system.message", "content": [{"type": "text", "text": "The user is in Madrid (Europe/Madrid timezone)."}]},
        ],
    )
    follow(stream)

[user] What's my local time right now? Check the sandbox clock with bash.


[bash] date; TZ=Europe/Madrid date


Agent: It's **16:57 (CEST, UTC+2) on Sunday, October 11, 2026** in Madrid. The sandbox clock reads 14:57 UTC.
[usage] $0.01 of $2.00, 5s active
[idle: end_turn]


## 5. Interrupt and Redirect

To change direction while the agent is working, send **`user.interrupt`** followed by a new **`user.message`**, in one request. The call returns as soon as both are queued; then:

1. The agent applies the interrupt. A model response in progress stops right away; running tool calls may take a moment.
2. `user.interrupt` appears on the stream, and **the interrupted turn ends with a `session.status_idle` whose `stop_reason` is `end_turn`** — the same value as a normal finish.
3. The agent starts a new turn with your redirect message.

Step 2 is the trap: if your loop stops at the first `end_turn`, it stops on the *interrupted* turn and never sees the redirect. So after an interrupt, count only an `end_turn` that comes **after your redirect message is processed** — that's what `follow(after_text=...)` does.

Below, we ask for a slow, ten-step job and interrupt it as soon as the first tool call shows up.

In [6]:
REDIRECT = "Stop that. Instead, just tell me the Python version installed in the sandbox."
interrupted = False


def interrupt_on_first_tool(event) -> None:
    global interrupted
    if event.type == "agent.tool_use" and not interrupted:
        interrupted = True
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.interrupt"}, message(REDIRECT)],
        )
        print(">>> sent user.interrupt + redirect")


with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[message(
            "Use bash to compute the SHA-256 hash of each of the strings 'a' through 'j', "
            "one separate command per string, and report all ten."
        )],
    )
    follow(stream, after_text=REDIRECT, on_event=interrupt_on_first_tool)

[user] Use bash to compute the SHA-256 hash of each of the strings 'a' through 'j', one


>>> sent user.interrupt + redirect
[bash] printf 'a' | sha256sum
[bash] printf 'b' | sha256sum
[bash] printf 'c' | sha256sum
[bash] printf 'd' | sha256sum
[bash] printf 'e' | sha256sum
[bash] printf 'f' | sha256sum
[bash] printf 'g' | sha256sum
[bash] printf 'h' | sha256sum
[bash] printf 'i' | sha256sum
[bash] printf 'j' | sha256sum
[interrupt applied]


[usage] $0.02 of $2.00, 11s active
[idle: end_turn — the interrupted turn; waiting for the redirect]


[user] Stop that. Instead, just tell me the Python version installed in the sandbox.


[bash] python3 --version; python --version 2>&1


Agent: The sandbox has **Python 3.11.16**, and both `python3` and `python` point to it.

I've stopped the hashing. Those ten commands were interrupted before they ran, so I don't have any hashes for you. My last reply also just repeated the error message, which wasn't useful.
[usage] $0.03 of $2.00, 17s active
[idle: end_turn]


Read the trace: the tool calls the agent had already requested **never ran** (there are no results for them), the interrupted turn ended with an `end_turn` idle that `follow()` skipped, and the redirect ran as a fresh turn.

> **Interrupt vs. a plain follow-up.** A `user.message` sent while the agent is running doesn't cut in: it's queued (its `processed_at` stays null) until the agent picks it up. Interrupt first when you need the agent to stop *now*.

## 6. Reconnect Without Missing Events

Long sessions outlive client connections: a notebook kernel restarts, a web server redeploys, a laptop sleeps. The session keeps working server-side regardless — but **a new stream only delivers events emitted after it opens**.

To reconnect without losing anything:

1. **Open a new stream** (it starts buffering).
2. **List the event history** to catch up, and remember every event `id` you've seen.
3. **Tail the stream**, skipping events already seen in the history.

To simulate a drop, we send a message *without* any stream open, wait a few seconds while the agent works unobserved, then reconnect.

In [7]:
sent = client.beta.sessions.events.send(
    session.id,
    events=[message("Use bash to count the entries in /etc, then tell me the count.")],
)
my_message_id = sent.data[0].id
print(f"Sent {my_message_id} with no stream open — 'disconnected' for 5 seconds...")
time.sleep(5)

Sent sevt_0196PoUaGSFUooeGfFG794XP with no stream open — 'disconnected' for 5 seconds...


In [8]:
def print_event(event) -> None:
    if event.type == "agent.message":
        print(f"Agent: {text_of(event)}")
    elif event.type == "agent.tool_use":
        print(f"[{event.name}] {str(event.input.get('command') or '')[:90]}")


with client.beta.sessions.events.stream(session.id) as stream:            # 1. open the stream first
    history = list(client.beta.sessions.events.list(session.id))           # 2. catch up from history
    seen = {event.id for event in history}

    # Replay what happened since our message, while we were away
    start = next((i for i, event in enumerate(history) if event.id == my_message_id), len(history))
    missed = history[start + 1:]
    print(f"--- caught up {len(missed)} events from history ---")
    for event in missed:
        print_event(event)

    # Done only if an end_turn idle came AFTER our message (an older idle doesn't count)
    finished = any(e.type == "session.status_idle" and e.stop_reason.type == "end_turn" for e in missed)
    if finished:
        print("--- the turn finished while we were away ---")
    else:
        print("--- now tailing the live stream ---")
        for event in stream:                                               # 3. tail, skipping duplicates
            if event.id in seen:
                continue
            seen.add(event.id)
            print_event(event)
            if event.type == "session.status_idle" and event.stop_reason.type != "requires_action":
                print(f"[idle: {event.stop_reason.type}]")
                break

--- caught up 10 events from history ---
[bash] ls -A /etc | wc -l
Agent: `/etc` has **160** entries. That counts hidden files but not subdirectory contents.
--- the turn finished while we were away ---


Either way you get every event exactly once: whatever happened while disconnected comes from the history, and the rest from the live stream.

## 7. Audit with the Event History

The history is also your audit log. `events.list()` takes a **`types`** filter. Here: every command the agent ran in this session, in order. The SDK iterator fetches every page for you.

In [9]:
for event in client.beta.sessions.events.list(session.id, types=["agent.tool_use"]):
    print(f"{event.processed_at:%H:%M:%S}  [{event.name}] {str(event.input.get('command') or '')[:80]}")

14:57:05  [bash] date; TZ=Europe/Madrid date
14:57:15  [bash] printf 'a' | sha256sum
14:57:15  [bash] printf 'b' | sha256sum
14:57:15  [bash] printf 'c' | sha256sum
14:57:15  [bash] printf 'd' | sha256sum
14:57:15  [bash] printf 'e' | sha256sum
14:57:15  [bash] printf 'f' | sha256sum
14:57:15  [bash] printf 'g' | sha256sum
14:57:15  [bash] printf 'h' | sha256sum
14:57:15  [bash] printf 'i' | sha256sum
14:57:15  [bash] printf 'j' | sha256sum
14:57:17  [bash] python3 --version; python --version 2>&1
14:57:23  [bash] ls -A /etc | wc -l


## 8. Pause at the Budget, Then Resume

Let's make the session hit its cap. We read the current list cost from **`sessions.retrieve()`**, then lower the budget to just above it. A new cap must be **strictly greater** than what the session has already consumed.

The budget is checked **between model requests**. The request that crosses the cap still finishes, so the final cost can land a little past it. Then every thread pauses, and the session goes idle with **`stop_reason: budget_reached`** — preceded, as always, by a `session.usage` event.

While paused, the session only accepts events that settle work in progress (tool results, confirmations, interrupts). A new `user.message` is rejected with a 400 error. To continue, **change or remove the budget**: the paused work resumes on its own, with no event from you. Since that resume is a server-side action, we keep the **same stream open** across the update — stream-first again.

In [10]:
consumed = int(client.beta.sessions.retrieve(session.id).usage.list_cost.amount)
client.beta.sessions.update(
    session.id,
    budget={"type": "limit", "max_list_cost": {"amount": str(consumed + 1), "currency": "USD"}},
)
print(f"Consumed so far: {consumed} cents -> budget lowered to {consumed + 1} cents")

Consumed so far: 4 cents -> budget lowered to 5 cents


In [11]:
TASK = (
    "Work in small steps, one bash command per step, and look at each result before the next step: "
    "1) create /tmp/lab, 2) write a Python script there that prints the first 20 primes, 3) run it, "
    "4) write a second script that prints the first 20 Fibonacci numbers, 5) run it, "
    "6) list /tmp/lab. Then summarize."
)

with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(session.id, events=[message(TASK)])
    reason = follow(stream)

    if reason == "budget_reached":
        paused = client.beta.sessions.retrieve(session.id)
        print(f"\n>>> paused at {cents(paused.usage.list_cost)} — raising the budget by $1.00\n")
        client.beta.sessions.update(
            session.id,
            budget={"type": "limit", "max_list_cost": {
                "amount": str(int(paused.usage.list_cost.amount) + 100), "currency": "USD"}},
        )
        follow(stream)   # same stream: the paused work resumes by itself

[user] Work in small steps, one bash command per step, and look at each result before t


[bash] mkdir -p /tmp/lab && ls -ld /tmp/lab


[bash] cat > /tmp/lab/primes.py <<'EOF'
primes = []
n = 2
while len(primes) < 20:
    if all(n % 


[usage] $0.05 of $0.05, 29s active
[idle: budget_reached]



>>> paused at $0.05 — raising the budget by $1.00



[bash] python3 /tmp/lab/primes.py


[bash] cat > /tmp/lab/fib.py <<'EOF'
a, b = 0, 1
fibs = []
for _ in range(20):
    fibs.append(a)


[bash] python3 /tmp/lab/fib.py


[bash] ls -la /tmp/lab


Agent: All six steps worked, one command each:

1. I created `/tmp/lab`.
2. I wrote `primes.py`.
3. Running it printed the first 20 primes: 2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71.
4. I wrote `fib.py`.
5. Running it printed the first 20 Fibonacci numbers, starting from 0: 0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181.
6. `/tmp/lab` now holds just the two scripts: `fib.py` (93 bytes) and `primes.py` (141 bytes).
[usage] $0.08 of $1.05, 41s active
[idle: end_turn]


If the cap was crossed by the very request that finished the agent's work, the session still reports `budget_reached`, and raising the budget then finds nothing left to do. The run usually pauses earlier in this task, though, and the resumed turn picks up where it stopped.

> In a session with many agents (modules 06–07), **one budget covers all of them**, and workflow runs pause at the cap too. That's why module 07 sets one from the start.

## 9. Inspect, List, Archive

- **`sessions.retrieve()`** returns the session's `status` (`idle`, `running`, `rescheduling`, `terminated`) and its cumulative `usage`.
- **`sessions.list()`** pages through your sessions.
- **`sessions.archive()`** makes a session read-only: no new events, history kept. A `running` session can't be archived — interrupt it and wait for `idle` first.
- **`sessions.delete()`** removes the session, its events, its sandbox, and **the output files it produced**. Download what you need first.

A session's sandbox is checkpointed whenever it goes idle, and a later `user.message` resumes it — even days later. The sandbox state is kept for 30 days from its creation.

In [12]:
final = client.beta.sessions.retrieve(session.id)
u = final.usage
print(f"Status        : {final.status}")
print(f"List cost     : {cents(u.list_cost)}  (budget {cents(final.budget.max_list_cost)})")
print(f"Tokens        : {u.input_tokens} in, {u.output_tokens} out, {u.cache_read_input_tokens} cache reads")
print(f"Active time   : {u.active_seconds:.0f}s")

print("\nRecent sessions:")
for s in client.beta.sessions.list(limit=3).data:   # .data = this page only; iterating the page would fetch them all
    print(f"  {s.id}  {s.status:<10} {s.title}")

Status        : idle
List cost     : $0.08  (budget $1.05)
Tokens        : 60 in, 2075 out, 106471 cache reads
Active time   : 41s

Recent sessions:
  sesn_01Kk66ZmWsXn8GFjCzBLxQff  idle       Session control
  sesn_01JNDY13gnnLknLn5aBawhi2  idle       Session control
  sesn_017UteVzEEhNK9FyRbMuyc4h  idle       Contract review CLI


In [13]:
archived = client.beta.sessions.archive(session.id)
print(f"Archived at {archived.archived_at}")

Archived at 2026-10-11 14:57:49.877928+00:00


## Summary

```
sessions.create(budget={type: limit, max_list_cost: {amount: "200", currency: USD}})
session.usage event          →  cumulative list cost, right before every idle
system.message               →  add system-level context mid-session
user.interrupt + user.message →  stop and redirect; skip the interrupted turn's end_turn
reconnect                    →  open stream → list history (seen ids) → tail, skipping seen
events.list(types=[...])     →  audit log
budget_reached               →  sessions.update(budget=...) resumes the paused work
sessions.retrieve / list / archive / delete
```

Key takeaways:
- Budgets are **hard caps** on list cost, set at creation, adjustable later; a session at its cap pauses, it doesn't die
- An interrupt ends the turn with a normal-looking `end_turn` — wait for the redirect before you stop reading
- A new stream has no memory: catch up from the history, then tail and de-duplicate
- One session, one budget — even when it runs many agents

### Next steps
- **06-subagents** — delegate to other agents in session threads
- **07-dynamic-workflows** — let the agent write a workflow that runs many agents in parallel